# Datasets

> download and store datasets

In [1]:
#| default_exp datasets

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
import os
# import json
from pathlib import Path

import quilt3
import pandas as pd
import numpy as np

### Download data via Quilt/T4

Allen Institute Cell Science (AICS)

In [4]:
#| export

def aics_pipeline(n_images_to_download=40, image_save_dir=None): 
    # Set default save directory if not provided
    if image_save_dir is None:
        image_save_dir = os.getcwd()

    # Ensure the save directory exists; create it if not
    os.makedirs(image_save_dir, exist_ok=True)

    # Load the package
    package = quilt3.Package.browse("aics/pipeline_integrated_cell", registry="s3://allencell")
    
    # Load metadata
    data_manifest = package["metadata.csv"]()

    # Keep only unique FOVs
    unique_fov_indices = np.unique(data_manifest['FOVId'], return_index=True)[1]
    data_manifest = data_manifest.iloc[unique_fov_indices]

    # Select first n_images_to_download
    data_manifest = data_manifest.iloc[:n_images_to_download]

    # Get source and target paths
    image_source_paths = data_manifest["SourceReadPath"]
    image_target_paths = [os.path.join(image_save_dir, os.path.basename(image_source_path)) 
                          for image_source_path in image_source_paths]

    # Download images
    downloaded_image_paths = []
    for image_source_path, image_target_path in zip(image_source_paths, image_target_paths):
        if os.path.exists(image_target_path):
            continue  # Skip if already downloaded
        
        try:
            package[image_source_path].fetch(image_target_path)
            downloaded_image_paths.append(image_target_path)
        except (OSError, FileNotFoundError) as e:
            print(f"Failed to fetch {image_source_path}: {e}")
                
    return downloaded_image_paths, data_manifest


In [5]:
image_target_paths, data_manifest = aics_pipeline(1, "../_data/aics")


Loading manifest: 100%|██████████| 77165/77165 [00:01<00:00, 46.5k/s]
100%|██████████| 372M/372M [00:22<00:00, 16.9MB/s]  


In [6]:
print(image_target_paths)
data_manifest #.to_csv('aics_dataset.csv')

['../_data/aics/6677e50c_3500001004_100X_20170623_5-Scene-1-P24-E06.ome.tiff']


,ProteinDisplayName,StructureSegmentationAlgorithmVersion,WorkflowId,NucMembSegmentationAlgorithm,CellIndex,Gene,WellId,StructureShortName,NucMembSegmentationAlgorithmVersion,WellName,...,Clone,Col,StructureDisplayName,DataSetId,ChannelNumber638,ChannelNumberBrightfield,PlateId,StructEducationName,SourceReadPath,FeatureExplorerURL
4131,Tom20,51,1,Matlab nucleus/membrane segmentation,1,TOMM20,24822,Mitochondria,1.3.0,E6,...,27,5,Mitochondria,3,1,6,3500001004,NaN,fovs/6677e50c_3500001004_100X_20170623_5-Scene...,https://cfe.allencell.org/?selectedPoint[0]=18...


### Dataset Manifest

In [7]:
# Make a manifest of all of the files in csv form

# df = pd.DataFrame(columns=["path_tiff", "channel_signal", "channel_target"])

# df["path_tiff"] = image_target_paths
# df["channel_signal"] = data_manifest["ChannelNumberBrightfield"].values
# df["channel_target"] = data_manifest[
#     "ChannelNumber405"
# ].values  # this is the DNA channel for all FOVs

# n_train_images = int(n_images_to_download * train_fraction)
# df_train = df[:n_train_images]
# df_test = df[n_train_images:]

# df_test.to_csv(data_save_path_test, index=False)
# df_train.to_csv(data_save_path_train, index=False)

In [8]:
#| hide
import nbdev; nbdev.nbdev_export()